In [11]:
import os
import sys

project_root = r"C:\Users\Kreesh\OneDrive\Desktop\Rippling\interneers-lab\backend\python"

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added to sys.path")
print(sys.path[-1])

Project root added to sys.path
C:\Users\Kreesh\OneDrive\Desktop\Rippling\interneers-lab\backend\python


In [12]:
import os
import json
from decimal import Decimal
from dotenv import load_dotenv
from google import genai
from week4.db_connection import initialize_mongo
from week4.models import Product, ProductCategory
from week6.schema import ProductListSchema

In [13]:
load_dotenv()
initialize_mongo()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Gemini and MongoDB initialized successfully")

Gemini and MongoDB initialized successfully


In [14]:
prompt = """
Generate exactly 5 products for a toy store.

Return valid JSON only in this structure:
{
  "products": [
    {
      "name": "string",
      "description": "string",
      "category": "string",
      "brand": "string",
      "price": 10.99,
      "quantity": 25
    }
  ]
}

Rules:
- Include exactly 5 products
- Product name must never be empty
- Brand must never be empty
- Description must never be empty
- Use realistic toy store categories like puzzles, dolls, action figures, board games, educational toys, building blocks, plush toys, remote control toys, outdoor toys
- price must be a float greater than 0
- quantity must be an integer greater than or equal to 0
- do not include markdown
- output valid JSON only
"""

In [15]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "temperature": 0.3,
        "response_mime_type": "application/json",
        "response_schema": ProductListSchema,
    }
)

print(response.text[:2000])

{"products":[{"name":"Starlight Princess Doll","description":"A beautiful doll with shimmering gown and magical accessories, perfect for imaginative play.","category":"Dolls","brand":"Dreamland Toys","price":29.99,"quantity":50},{"name":"Galactic Warrior Action Figure","description":"Highly detailed action figure with multiple points of articulation and interchangeable weapons for epic space battles.","category":"Action Figures","brand":"Cosmic Play","price":19.50,"quantity":75},{"name":"Enchanted Forest Jigsaw Puzzle","description":"A 1000-piece jigsaw puzzle featuring a vibrant illustration of a magical forest scene, challenging and fun for all ages.","category":"Puzzles","brand":"Mind Bender Puzzles","price":15.75,"quantity":30},{"name":"Robo-Racer Remote Control Car","description":"High-speed remote control car with durable design and responsive controls, ideal for indoor and outdoor racing.","category":"Remote Control Toys","brand":"Speedy Wheels","price":45.00,"quantity":20},{"na

In [16]:
validated_products = ProductListSchema.model_validate_json(response.text)
print("Validated product count:", len(validated_products.products))

Validated product count: 5


In [17]:
product_preview = [product.model_dump() for product in validated_products.products[:5]]
product_preview

[{'name': 'Starlight Princess Doll',
  'description': 'A beautiful doll with shimmering gown and magical accessories, perfect for imaginative play.',
  'category': 'Dolls',
  'brand': 'Dreamland Toys',
  'price': 29.99,
  'quantity': 50},
 {'name': 'Galactic Warrior Action Figure',
  'description': 'Highly detailed action figure with multiple points of articulation and interchangeable weapons for epic space battles.',
  'category': 'Action Figures',
  'brand': 'Cosmic Play',
  'price': 19.5,
  'quantity': 75},
 {'name': 'Enchanted Forest Jigsaw Puzzle',
  'description': 'A 1000-piece jigsaw puzzle featuring a vibrant illustration of a magical forest scene, challenging and fun for all ages.',
  'category': 'Puzzles',
  'brand': 'Mind Bender Puzzles',
  'price': 15.75,
  'quantity': 30},
 {'name': 'Robo-Racer Remote Control Car',
  'description': 'High-speed remote control car with durable design and responsive controls, ideal for indoor and outdoor racing.',
  'category': 'Remote Contro

In [18]:
def get_or_create_category(category_title):
    cat_title = category_title.strip()

    if not cat_title:
        cat_title = "Miscellaneous"

    category = ProductCategory.objects(title=cat_title).first()
    if category:
        return category

    category = ProductCategory(
        title=cat_title,
        description=f"Auto-created from Gemini synthetic inventory data for {cat_title}"
    )
    category.save()
    return category

In [19]:
saved_count = 0

for item in validated_products.products:
    new_name = item.name.strip()
    new_brand = item.brand.strip()
    new_description = item.description.strip()
    new_category = item.category.strip()
    new_price = item.price
    new_quantity = item.quantity
    if not new_name:
        print("Skipped one product because name was empty after stripping.")
        continue

    if not new_brand:
        print(f"Skipped product '{new_name}' because brand was empty after stripping.")
        continue

    if not new_description:
        print(f"Skipped product '{new_name}' because description was empty after stripping.")
        continue

    if new_price is None or new_price <= 0:
        print(f"Skipped product '{new_name}' because price was invalid: {new_price}")
        continue

    if new_quantity is None or new_quantity < 0:
        print(f"Skipped product '{new_name}' because quantity was invalid: {new_quantity}")
        continue

    category_obj = get_or_create_category(new_category)

    product = Product(
        name=new_name,
        description=new_description,
        price=Decimal(str(item.price)),
        brand=new_brand,
        quantity=int(item.quantity),
        category=category_obj
    )
    product.save()
    saved_count += 1

print("Saved products:", saved_count)

Saved products: 5


In [20]:
import os
os.makedirs("week6", exist_ok=True)

with open("week6/generated_products.json", "w", encoding="utf-8") as file:
    json.dump(
        {"products": [product.model_dump() for product in validated_products.products]},
        file,
        indent=2
    )

print("Exported validated products to week6/generated_products.json")

Exported validated products to week6/generated_products.json
